# Project 01 (basic) — A STRIPS forward planner

**Module 07 — Theory of AI 2**

You build a **classical planner**: STRIPS states and actions, forward search
(progression) with BFS *and* with A\* under the **h_add heuristic** from the
delete relaxation (script part 1). The test case is the famous **Sussman
anomaly** of the blocks world — a small problem on which naive subgoal planners
fail, but state search does not.

The goal: to understand how planning can be formulated as search (module 06) and
how a **heuristic obtained automatically from the action description** guides the
search.

## Setup
Only the standard library. Select the kernel of the repository `.venv` (see
`SETUP.md`) and run the cells from top to bottom. Then solve **tasks 1–3** (the
`TODO` cells).

In [1]:
import heapq, itertools
from collections import deque
from dataclasses import dataclass, field
from typing import FrozenSet, Tuple, Any
print("Libraries loaded (only the standard library).")

Libraries loaded (only the standard library).


## Part A — STRIPS and the blocks world (given)
The representation: fluents as tuples, states as a `frozenset`, actions as
`Action(pre, add, dele)`. Progression is pure set arithmetic
$(s\setminus\mathrm{DEL})\cup\mathrm{ADD}$. Read the cells and run them.

In [2]:
# ---- The STRIPS representation ---------------------------------------------
# A fluent is a tuple, e.g. ("on","C","A") or ("handempty",).
# A state is a frozenset of fluents (closed world: whatever is missing is false).

@dataclass(frozen=True)
class Action:
    name: str
    pre: FrozenSet
    add: FrozenSet
    dele: FrozenSet          # "del" is a Python keyword -> dele
    def __repr__(self): return self.name

def applicable(action, state):
    """Applicable if all preconditions hold in the state."""
    return action.pre <= state

def result(state, action):
    """Progression:  (s without DEL) united with ADD."""
    return frozenset((state - action.dele) | action.add)

# ---- Blocks world: instantiate the action schemas for concrete blocks -------
def blocksworld_actions(blocks):
    acts = []
    for x in blocks:
        acts.append(Action(f"PickUp({x})",
            pre=frozenset({("clear", x), ("ontable", x), ("handempty",)}),
            add=frozenset({("holding", x)}),
            dele=frozenset({("clear", x), ("ontable", x), ("handempty",)})))
        acts.append(Action(f"PutDown({x})",
            pre=frozenset({("holding", x)}),
            add=frozenset({("ontable", x), ("clear", x), ("handempty",)}),
            dele=frozenset({("holding", x)})))
        for y in blocks:
            if x == y:
                continue
            acts.append(Action(f"Stack({x},{y})",
                pre=frozenset({("holding", x), ("clear", y)}),
                add=frozenset({("on", x, y), ("clear", x), ("handempty",)}),
                dele=frozenset({("holding", x), ("clear", y)})))
            acts.append(Action(f"Unstack({x},{y})",
                pre=frozenset({("on", x, y), ("clear", x), ("handempty",)}),
                add=frozenset({("holding", x), ("clear", y)}),
                dele=frozenset({("on", x, y), ("clear", x), ("handempty",)})))
    return acts

@dataclass(frozen=True)
class PlanProblem:
    initial: FrozenSet
    goal: FrozenSet
    actions: Tuple
    def is_goal(self, state): return self.goal <= state
    def successors(self, state):
        return [(a, result(state, a)) for a in self.actions if applicable(a, state)]

# ---- The Sussman anomaly (the famous blocks world test case) ---------------
BLOCKS = ["A", "B", "C"]
s0 = frozenset({("on", "C", "A"), ("ontable", "A"), ("ontable", "B"),
                ("clear", "C"), ("clear", "B"), ("handempty",)})
goal = frozenset({("on", "A", "B"), ("on", "B", "C")})
problem = PlanProblem(s0, goal, tuple(blocksworld_actions(BLOCKS)))

def show_state(s):
    parts = []
    for f in sorted(s):
        parts.append(f[0] + ("(" + ",".join(f[1:]) + ")" if len(f) > 1 else ""))
    return "  ".join(parts)

print("Start :", show_state(s0))
print("Goal  :", show_state(goal))
print("Ground actions:", len(problem.actions))

Start : clear(B)  clear(C)  handempty  on(C,A)  ontable(A)  ontable(B)
Goal  : on(A,B)  on(B,C)
Ground actions: 18


## Part B — The search infrastructure and BFS (given)
The same `Node` as in module 06 and the generic best-first search. BFS is given
completely, as a model.

In [3]:
# ---- Search nodes (as in module 06) ----------------------------------------
@dataclass(frozen=True)
class Node:
    state: Any
    parent: Any = None
    action: Any = None
    path_cost: float = 0.0     # = the number of actions up to here
    def plan(self):
        acts, n = [], self
        while n.parent is not None:
            acts.append(n.action); n = n.parent
        return list(reversed(acts))

print("Search nodes ready.")

Search nodes ready.


In [4]:
# ---- Forward search with BFS (given completely) ----------------------------
def bfs_plan(problem):
    node = Node(problem.initial)
    if problem.is_goal(node.state):
        return node, 0
    frontier = deque([node]); reached = {node.state}; expanded = 0
    while frontier:
        node = frontier.popleft(); expanded += 1
        for a, s2 in problem.successors(node.state):
            if s2 in reached:
                continue
            child = Node(s2, node, a, node.path_cost + 1)
            if problem.is_goal(s2):
                return child, expanded
            reached.add(s2); frontier.append(child)
    return None, expanded

node, exp = bfs_plan(problem)
print("BFS plan (", len(node.plan()), "actions ), expanded states:", exp)
for i, a in enumerate(node.plan(), 1):
    print(f"  {i}. {a}")

BFS plan ( 6 actions ), expanded states: 18
  1. Unstack(C,A)
  2. PutDown(C)
  3. PickUp(B)
  4. Stack(B,C)
  5. PickUp(A)
  6. Stack(A,B)


In [5]:
# ---- Generic best-first search (given) -------------------------------------
def best_first_plan(problem, f):
    node = Node(problem.initial)
    counter = itertools.count()
    frontier = [(f(node.state, 0), next(counter), node)]
    reached = {problem.initial: 0}
    expanded = 0
    while frontier:
        _, _, node = heapq.heappop(frontier)
        if problem.is_goal(node.state):
            return node, expanded
        expanded += 1
        for a, s2 in problem.successors(node.state):
            g2 = node.path_cost + 1
            if s2 not in reached or g2 < reached[s2]:
                reached[s2] = g2
                child = Node(s2, node, a, g2)
                heapq.heappush(frontier, (f(s2, g2), next(counter), child))
    return None, expanded

print("best_first_plan ready — A* is the right choice of f.")

best_first_plan ready — A* is the right choice of f.


### Task 1 — the h_add heuristic (delete relaxation)
Implement `h_add`: ignore all delete lists and estimate the cost of every fluent
by a fixed-point iteration; sum over the goal fluents. See the formula in the
cell comment and script part 1.4.

In [6]:
# ---- Delete relaxation: the h_add heuristic --------------------------------
def h_add(state, problem):
    """The estimated cost of reaching the goal while ignoring all delete lists.
    Delta(p)=0 for p in state, otherwise the min over actions a with p in ADD(a)
    of (1 + sum_{q in PRE(a)} Delta(q)). A fixed-point iteration.
    h_add = the sum of the Delta over the goal fluents."""
    INF = float("inf")
    delta = {f: 0 for f in state}
    changed = True
    while changed:
        changed = False
        for a in problem.actions:
            if any(delta.get(q, INF) == INF for q in a.pre):
                continue
            cost_a = 1 + sum(delta[q] for q in a.pre)
            for p in a.add:
                if cost_a < delta.get(p, INF):
                    delta[p] = cost_a; changed = True
    total = sum(delta.get(g, INF) for g in problem.goal)
    return total

print("h_add(start) =", h_add(problem.initial, problem))

h_add(start) = 5


### Task 2 — A\* with h_add
Wire `best_first_plan` up with $f(s,g)=g+h_{\text{add}}(s)$ to get A\*.

In [7]:
# ---- A* with h_add ---------------------------------------------------------
def astar_plan(problem):
    return best_first_plan(problem, f=lambda s, g: g + h_add(s, problem))

node, exp = astar_plan(problem)
print("A* plan (", len(node.plan()), "actions ), expanded states:", exp)
for i, a in enumerate(node.plan(), 1):
    print(f"  {i}. {a}")

A* plan ( 6 actions ), expanded states: 11
  1. Unstack(C,A)
  2. PutDown(C)
  3. PickUp(B)
  4. Stack(B,C)
  5. PickUp(A)
  6. Stack(A,B)


### Task 3 — the comparison
Compare BFS and A\*(h_add): the plan length and the number of expanded states.
Both should find an optimal **6-step plan**; A\* expands fewer.

In [8]:
# ---- Comparing BFS and A*(h_add) -------------------------------------------
b_node, b_exp = bfs_plan(problem)
a_node, a_exp = astar_plan(problem)
print(f"BFS       : plan length {len(b_node.plan())}, {b_exp} states expanded")
print(f"A*(h_add) : plan length {len(a_node.plan())}, {a_exp} states expanded")
assert len(b_node.plan()) == len(a_node.plan()) == 6, "the Sussman optimum is 6 actions"
print("\nBoth find an optimal 6-step plan; the heuristic saves expansions.")
print("(The Sussman anomaly: the goal does NOT decompose into independent subgoals —")
print(" a naive subgoal planner fails, state search does not.)")

BFS       : plan length 6, 18 states expanded
A*(h_add) : plan length 6, 11 states expanded

Both find an optimal 6-step plan; the heuristic saves expansions.
(The Sussman anomaly: the goal does NOT decompose into independent subgoals —
 a naive subgoal planner fails, state search does not.)


## Part C — Verification
Finally we check that the plan found is applicable step by step and reaches the
goal.

In [9]:
# ---- Self-check: is the A* plan really executable and goal-reaching? -------
s = problem.initial
for a in a_node.plan():
    assert applicable(a, s), f"the action {a} is not applicable!"
    s = result(s, a)
assert problem.is_goal(s), "the final state does not satisfy the goal!"
print("Verified: the plan is executable step by step and reaches the goal.")
print("Final state:", show_state(s))

Verified: the plan is executable step by step and reaches the goal.
Final state: clear(A)  handempty  on(A,B)  on(B,C)  ontable(C)


## Reflection — reference answers

1. **Why the Sussman anomaly is hard for subgoal decomposition.** The two
   subgoals `On(A,B)` and `On(B,C)` are not independent — they *interfere*. If
   you first achieve `On(A,B)`, then A sits on B, so B is no longer clear and you
   cannot put B on C without dismantling what you just built. If you first
   achieve `On(B,C)`, you must first get C off A, which is fine, but then placing
   A on B requires A to be clear and held — the orders interlock. Any planner
   that solves the subgoals *sequentially and independently* produces a plan that
   is either longer than 6 steps or has to undo its own work. The state search
   does not care: it simply searches the space of complete states, where the
   interleaving `Unstack(C,A) → PutDown(C) → PickUp(B) → Stack(B,C) → PickUp(A) →
   Stack(A,B)` is just one path among others. Historically this example is
   exactly what killed the naive "linear planners".
2. **h_add is inadmissible — why it does not hurt here.** $h_{\text{add}}$ sums
   the estimated cost of every goal fluent separately, so it counts shared
   subplans several times and can overestimate the true remaining cost. With an
   inadmissible heuristic A\* loses its *guarantee* of optimality — it may commit
   to a goal node before a cheaper path has been explored. Here it happens to
   return the optimum, and we do not have to take that on trust: the assert in
   task 3 compares against BFS, which *is* optimal at unit costs, and both give 6.
   It would hurt as soon as the overestimation makes the cheap path look worse
   than an expensive one. If you need the guarantee, use $h_{\max}$ (the maximum
   instead of the sum), which is admissible — at the price of being much weaker
   and expanding considerably more states.
3. **What would change with regression.** You would search backwards from the
   goal description, and a "state" would no longer be a complete set of fluents
   but a *partial* description — the set of conditions that still have to hold.
   The step is $\mathrm{Regress}(g',a)=(g'\setminus\mathrm{ADD}(a))\cup\mathrm{PRE}(a)$
   for actions that are relevant ($\mathrm{ADD}(a)\cap g'\neq\emptyset$) and
   consistent ($\mathrm{DEL}(a)\cap g'=\emptyset$). The advantage is a much
   smaller branching factor, because only actions that contribute to the goal are
   considered at all. The price: heuristics are harder to define on partial
   descriptions, and one can generate condition sets that no reachable state
   satisfies.
4. **Where h_add's domain knowledge comes from.** It is not built into the
   heuristic — it is read out of the STRIPS description at runtime. `h_add`
   only ever looks at `a.pre` and `a.add` of the given actions and computes the
   fixed point from them. That is exactly what "domain-independent" means: the
   same code delivers a sensible heuristic for the blocks world, for logistics or
   for a lift domain, because the *domain* is data (the action schemas), not
   code. That is the reason modern planners can accept a PDDL file and get going
   without any hand-written knowledge about the domain.